In [ ]:
import torch
import torch.nn as nn
from torch.nn import CrossEntropyLoss
from transformers import BertTokenizer, BertModel
from tqdm import tqdm
import pandas as pd

In [3]:
class Bert(nn.Module):
    def __init__(self, bert_base_model_dir, label_size, drop_out_rate = 0.5):
        super(Bert,self).__init__()
        self.label_size = label_size

        #self.device = torch.device('cuda')

        self.bert_tokenizer = BertTokenizer.from_pretrained(bert_base_model_dir) 
        self.bert_model = BertModel.from_pretrained(bert_base_model_dir)

        self.dropout=nn.Dropout(drop_out_rate)#全连接分类层
        self.cls_layer = nn.Linear(self.bert_model.config.hidden_size, label_size)

    def forward(self, input_ids, attention_mask, token_type_ids=None, position_ids=None, head_mask=None, labels=None):
        last_hidden_state, pooled_output = self.bert_model(
            input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids, 
            position_ids=position_ids, head_mask=head_mask, inputs_embeds=None, return_dict=False
        )
        cls_outs = self.dropout(pooled_output)
        logits = self.cls_layer(cls_outs)

        best_labels = torch.argmax(logits, dim=-1)
        #best_labels = best_labels.to(self.device)

        if labels is not None:
            loss = CrossEntropyLoss(ignore_index=-1)(logits, labels)#label=-1被忽略
            return best_labels, loss
        
        return best_labels#直接返回预测的labels
    def get_bert_tokenizer(self):
        return self.bert_tokenizer

In [4]:
device = torch.accelerator.current_accelerator().type

label_size = 6
model_path = "./bert-base-chinese"
model = Bert(model_path, label_size).to(device)
print(model)

Bert(
  (bert_model): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(21128, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_af

d:\APPs\Anaconda\Lib\site-packages\torch\cuda\__init__.py:287: UserWarning: 
NVIDIA GeForce RTX 5060 Laptop GPU with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_75 sm_80 sm_86 sm_89 sm_90 compute_90.
If you want to use the NVIDIA GeForce RTX 5060 Laptop GPU GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


In [5]:
loss_fn = nn.CrossEntropyLoss()
learning_rate = 2e-5

no_decay = ["bias", "LayerNorm.weight"]
optimizer_parameters = [
    {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay' : 0.01},
    {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay' : 0.0},
]
optimizer = torch.optim.AdamW(optimizer_parameters, lr=learning_rate)

tokenizer = BertTokenizer.from_pretrained(model_path)


In [6]:
def tokenize(texts):
    return tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )


In [7]:
def train(train_texts, labels, batch_size):
    model.train()
    total_loss = 0
    for i in range(0, len(train_texts), batch_size):
        batch_texts = train_texts[i:i+batch_size]
        batch_labels = labels[i:i+batch_size]
    
        encoded =  tokenize(batch_texts)

        # 把数据搬到 GPU 
        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)
        batch_label = torch.tensor(batch_labels, dtype=torch.long).to(device)


        # 前向传播（BERT 内部自动算 loss）
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=batch_label
        )

        #loss = loss_fn(outputs, batch_label)
        preds, loss = outputs

        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        batch_idx = i // batch_size
        if batch_idx % 100 == 0: # 这样才是每训练 100 个批次打印一次
            print(f"[{batch_idx}] loss: {loss.item():>7f}")


In [8]:
def test(test_texts, labels, batch_size):
    model.eval()
    test_loss, correct = 0, 0
    test_texts_size = len(test_texts)
    num_batches = 0
    with torch.no_grad():
        for i in range(0, test_texts_size, batch_size):
            batch_texts = test_texts[i : i + batch_size]
            batch_labels = labels[i : i + batch_size]
        
            encoded =  tokenize(batch_texts)

            # 把数据搬到 GPU 
            input_ids = encoded["input_ids"].to(device)
            attention_mask = encoded["attention_mask"].to(device)
            batch_label = torch.tensor(batch_labels, dtype=torch.long).to(device)

            # 前向传播（BERT 内部自动算 loss）
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=batch_label
            )

            preds, loss = outputs

            test_loss += loss.item() # 直接累加模型算好的 loss.item()
            correct += (preds == batch_label).sum().item() # 直接比较 preds 和真实标签
            num_batches += 1
    avg_loss = test_loss / num_batches
    accuracy = correct / test_texts_size   

    print(f"Test Result: \n Accuracy: {(100*accuracy):>0.1f}%, Avg loss: {avg_loss:>8f} \n")

In [9]:
df = pd.read_json('./data/weibo_for_train/usual_train.txt', encoding='utf-8')
print(df.head())


   id                                        content     label
0   1                  气死姐姐了，快二是阵亡了吗，尼玛，一个半小时过去了也没上车     angry
1   2               妞妞啊，今天又承办了一个发文登记文号是126~嘻~么么哒~晚安哟     happy
2   3               这里还值得注意另一个事实，就是张鞠存原有一个东溪草堂为其读书处。   neutral
3   4  这在前华约国家(尤其是东德)使用R-73的首次联合演习期间，被一些北约组织的飞行员所证实。   neutral
4   5                               TinyThief上wii了？！  surprise


In [10]:
label_map = {'happy': 0, 'angry': 1, 'sad': 2, 'fear': 3, 'surprise': 4, 'neutral': 5}
df['label'] = df['label'].map(label_map)


train_size = int(len(df) * 0.9)
train_text_data = df['content'].iloc[:train_size].tolist()
train_label_data = df['label'].iloc[:train_size].tolist()
test_text_data = df['content'].iloc[train_size:].tolist()
test_label_data = df['label'].iloc[train_size:].tolist()

In [11]:
epochs = 5
batch_size = 32
for t in range(epochs):
    print(f"========== Epoch {t+1} / {epochs} ==========")

    train(train_text_data, train_label_data, batch_size)
    test(test_text_data, test_label_data, batch_size)
    
    save_path = f"trained_bert_epoch_{t+1}.pth"
    torch.save(model.state_dict(), save_path)
    print(f"{t+1} model has saved: {save_path}\n")

print("Done")

========== Epoch 1 / 5 ==========


KeyboardInterrupt: 

========== Epoch 1 / 5 ==========
[0] loss: 0.056715
[100] loss: 0.109001
[200] loss: 0.162091
[300] loss: 0.056011
[400] loss: 0.463112
[500] loss: 0.063567
[600] loss: 0.511742
[700] loss: 0.076890
Test Result: 
 Accuracy: 75.6%, Avg loss: 0.978859 

1 model has saved: trained_bert_epoch_1.pth

========== Epoch 2 / 5 ==========
[0] loss: 0.111965
[100] loss: 0.055903
[200] loss: 0.083368
[300] loss: 0.089615
[400] loss: 0.156486
[500] loss: 0.021406
[600] loss: 0.299469
[700] loss: 0.053368
Test Result: 
 Accuracy: 75.7%, Avg loss: 1.060529 

2 model has saved: trained_bert_epoch_2.pth

========== Epoch 3 / 5 ==========
[0] loss: 0.064087
[100] loss: 0.022363
[200] loss: 0.218257
[300] loss: 0.015948
[400] loss: 0.117184
[500] loss: 0.061429
[600] loss: 0.080800
[700] loss: 0.036062
Test Result: 
 Accuracy: 75.9%, Avg loss: 1.149829 

3 model has saved: trained_bert_epoch_3.pth

========== Epoch 4 / 5 ==========
[0] loss: 0.013688
[100] loss: 0.032009
[200] loss: 0.026505
[300] loss: 0.038911
[400] loss: 0.020394
[500] loss: 0.031869
[600] loss: 0.036224
[700] loss: 0.127714
Test Result: 
 Accuracy: 76.6%, Avg loss: 1.169739 

4 model has saved: trained_bert_epoch_4.pth

========== Epoch 5 / 5 ==========
[0] loss: 0.050180
[100] loss: 0.056084
[200] loss: 0.184618
[300] loss: 0.006308
[400] loss: 0.081224
[500] loss: 0.004337
[600] loss: 0.016225
[700] loss: 0.005495
Test Result: 
 Accuracy: 76.3%, Avg loss: 1.177584 

5 model has saved: trained_bert_epoch_5.pth

Done
